In [ ]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
train_df = pd.read_csv('/content/drive/MyDrive/Datathon/train.csv')
test_df = pd.read_csv('/content/drive/MyDrive/Datathon/test_x.csv')

In [ ]:
# Feature Engineering (Özellik Mühendisliği) Fonksiyonu
def ozellikleri_uret(df):
    df_copy = df.copy()

    # 1. Kaliteli Uyku Yüzdesi
    if 'rem_yuzdesi' in df_copy.columns and 'derin_uyku_yuzdesi' in df_copy.columns:
        df_copy['toplam_kaliteli_uyku_yuzdesi'] = df_copy['rem_yuzdesi'] + df_copy['derin_uyku_yuzdesi']

    # 2. Zihinsel Yük Endeksi
    if 'stres_skoru' in df_copy.columns and 'gunluk_calisma_saati' in df_copy.columns:
        df_copy['zihinsel_yuk_endeksi'] = df_copy['stres_skoru'] * df_copy['gunluk_calisma_saati']

    # 3. Uyku Zorluğu Endeksi
    if 'gecelik_uyanma_sayisi' in df_copy.columns and 'uykuya_dalma_suresi_dk' in df_copy.columns:
        df_copy['uyku_zorlugu_endeksi'] = df_copy['gecelik_uyanma_sayisi'] * df_copy['uykuya_dalma_suresi_dk']

    # 4. Fiziksel Dinçlik Oranı (0'a bölünme hatasını önlemek için +0.1 ekliyoruz)
    if 'gunluk_adim_sayisi' in df_copy.columns and 'dinlenik_nabiz_bpm' in df_copy.columns:
        df_copy['fiziksel_dinclik_orani'] = df_copy['gunluk_adim_sayisi'] / (df_copy['dinlenik_nabiz_bpm'] + 0.1)

    return df_copy

In [ ]:
# Yeni özellikleri Train ve Test setine ekle
train_df = ozellikleri_uret(train_df)
test_df = ozellikleri_uret(test_df)

# Sadece kullanacağımız "Altın Kadro" sütunlarını seçiyoruz
secilecek_sutunlar = [
    'toplam_kaliteli_uyku_yuzdesi', 'zihinsel_yuk_endeksi',
    'uyku_zorlugu_endeksi', 'fiziksel_dinclik_orani',
    'meslek', 'ruh_sagligi_durumu', 'gun_tipi'
]

X_train_yeni = train_df[secilecek_sutunlar].copy()
y_train_yeni = train_df['bilissel_performans_skoru'].copy()
X_test_yeni = test_df[secilecek_sutunlar].copy()

kategorik_sutunlar = ['meslek', 'ruh_sagligi_durumu', 'gun_tipi']

In [ ]:
print("--- 2. Kategorik Veriler Olasılıksal Dağılıma Göre Dolduruluyor ---")
# Senin harika önerin: Eksik verileri orijinal dağılımı bozmadan doldurma
def dagilima_gore_doldur(train_data, test_data, col):
    # Sadece Train setinin dağılımını (olasılıklarını) öğreniyoruz
    olasiliklar = train_data[col].value_counts(normalize=True)
    kategoriler = olasiliklar.index
    oranlar = olasiliklar.values

    # Train'deki boşlukları olasılıklara göre rastgele doldur
    train_bos = train_data[col].isnull()
    if train_bos.sum() > 0:
        train_data.loc[train_bos, col] = np.random.choice(kategoriler, size=train_bos.sum(), p=oranlar)

    # Test'teki boşlukları Train'in olasılıklarına göre doldur
    test_bos = test_data[col].isnull()
    if test_bos.sum() > 0:
        test_data.loc[test_bos, col] = np.random.choice(kategoriler, size=test_bos.sum(), p=oranlar)

    return train_data, test_data

for cat_col in kategorik_sutunlar:
    X_train_yeni, X_test_yeni = dagilima_gore_doldur(X_train_yeni, X_test_yeni, cat_col)

In [ ]:
print("--- 3. Veriler Makinenin Anlayacağı Dile Çevriliyor (Encoding) ---")
# Sayısal imputer'ın kategorik verilerden faydalanabilmesi için önce One-Hot Encoding yapıyoruz
X_train_enc = pd.get_dummies(X_train_yeni, columns=kategorik_sutunlar, drop_first=True)
X_test_enc = pd.get_dummies(X_test_yeni, columns=kategorik_sutunlar, drop_first=True)

# Train ve Test setlerindeki sütunları hizalama
X_train_enc, X_test_enc = X_train_enc.align(X_test_enc, join='left', axis=1, fill_value=0)

In [ ]:
print("--- 4. Sayısal Veriler Iterative Imputer (MICE) İle Dolduruluyor ---")
# Sayısal verilerdeki eksiklikleri diğer sütunlara bakarak tahmin etme
imputer = IterativeImputer(random_state=42, max_iter=10)
# Fit işlemi sadece Train üzerinde!
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train_enc), columns=X_train_enc.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test_enc), columns=X_test_enc.columns)

In [ ]:
print("--- 5. Veriler Ölçeklendiriliyor (Robust Scaling) ---")
scaler = RobustScaler()
X_train_final = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=X_train_imp.columns)
X_test_final = pd.DataFrame(scaler.transform(X_test_imp), columns=X_test_imp.columns)

In [ ]:
print("--- 6. Şampiyon XGBoost Yeni Verilerle Eğitiliyor ---")
# Validation için ayır
X_tr, X_val, y_tr, y_val = train_test_split(X_train_final, y_train_yeni, test_size=0.2, random_state=42)

# Hiperparametre optimizasyonunda bulduğumuz en iyi ayarlar
best_xgb_params = {
    'subsample': 0.6,
    'n_estimators': 200,
    'max_depth': 3,
    'learning_rate': 0.1,
    'colsample_bytree': 0.6
}

xgb_model = XGBRegressor(**best_xgb_params, random_state=42, objective='reg:squarederror')
xgb_model.fit(X_tr, y_tr)

# Tahmin ve Metrikler
y_pred = xgb_model.predict(X_val)
yeni_rmse = np.sqrt(mean_squared_error(y_val, y_pred))
yeni_r2 = r2_score(y_val, y_pred)

print("*" * 50)
print(f"🚀 YENİ STRATEJİ İLE XGBOOST SONUÇLARI")
print(f"📉 Yeni Validation RMSE : {yeni_rmse:.4f}")
print(f"📈 Yeni Validation R²   : {yeni_r2:.4f}")
print("*" * 50)

--- 2. Kategorik Veriler Olasılıksal Dağılıma Göre Dolduruluyor ---
--- 3. Veriler Makinenin Anlayacağı Dile Çevriliyor (Encoding) ---
--- 4. Sayısal Veriler Iterative Imputer (MICE) İle Dolduruluyor ---
--- 5. Veriler Ölçeklendiriliyor (Robust Scaling) ---
--- 6. Şampiyon XGBoost Yeni Verilerle Eğitiliyor ---
**************************************************
🚀 YENİ STRATEJİ İLE XGBOOST SONUÇLARI
📉 Yeni Validation RMSE : 1.3533
📈 Yeni Validation R²   : 0.6353
**************************************************
